# Problem 44: Conversation Fork Handling

**Difficulty:** Hard | **Framework:** LangGraph

**Categories:** memory, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/conversation-fork-handling).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must support saving named checkpoints during conversation
- Restoring a checkpoint must revert the conversation state to that point
- New messages after restoration must continue from the restored state, not the latest
- The agent must list available checkpoints when asked


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import operator

llm = ChatOpenAI(model="gpt-4o-mini")

class State(TypedDict):
    messages: Annotated[list, operator.add]

# BUG: Linear conversation only — no branching or checkpoint support
def chatbot(state: State) -> State:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
app = graph.compile()

messages = [SystemMessage(content="You are a helpful trip planning assistant.")]

def chat(user_input: str) -> str:
    global messages
    messages.append(HumanMessage(content=user_input))
    result = app.invoke({"messages": messages})
    messages = result["messages"]
    return messages[-1].content

# Plan two options
print(chat("Plan me a 7-day trip. Option A: Japan. Option B: Italy."))
print(chat("Let's explore Option A — Japan. Give me a day-by-day itinerary."))
print(chat("Add a visit to Mount Fuji on day 3."))

# User wants to go back and explore Option B instead
# BUG: No way to restore state — the Japan itinerary is baked into history
print(chat("Actually, go back to where we had both options and let's explore Option B — Italy."))

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Store snapshots of the conversation history at labeled checkpoints (e.g., 'option_a', 'option_b')
2. When the user asks to 'go back to option A,' restore the history snapshot for that checkpoint and continue from there
3. Use a dictionary mapping checkpoint names to copies of the message history at that point in time


## 7. Evaluation Criteria

Check your solution against these criteria:

- Checkpoints can be saved
- Checkpoints can be restored
- Continues from restored state
- Can list available checkpoints
